### Conversational Buffer Memory : Older version first and then same with advance version using runnables

In [1]:
!pip install langchain langchain_community langchain_google_genai langchain_classic langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
import os

from google.colab import userdata
gem = userdata.get('gemini')

os.environ["GOOGLE_API_KEY"] = gem

In [3]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import LLMChain

In [ ]:
#Step 1 : Is to create a memory object (older version)

In [ ]:
memory1 = ConversationBufferMemory(memory_key = "storage",return_messages =True) #memory_object

# memory_key = "storage": act as a veriable(attribute) name to the Buffer memory


# ConversationBufferMemory :It is a class that stores the conversation history in a sequential manner.
# It maintains the interaction as a list of messages, continuously appending new inputs and responses as the conversation progresses.
# This allows the system to retain context across multiple turns, enabling more coherent and context-aware responses.

# memory_key is nothing but an object name like "history"=[1conv(Human_Message+AI_Message),2conv(Human_Message+AI_Message)]
# Human_Message+AI_Message this 2 are appended into list [Human_Message+AI_Message] and store in "history".

# return_messages = True:  ensures that the conversation history is returned in a structured format (i.e., as a list of message objects or dictionaries).
# - This allows you to work with individual messages more flexibly, such as accessing roles (`user`, `assistant`) and their corresponding content.
# - If set to **False**, the entire conversation history is returned as a single concatenated string instead of a structured list.
# In simple terms: True → Returns structured data as a list of dictionaries.(easy to process programmatically)
# False → Returns plain text (string) (easy to read, but less flexible)
# Problem : if return_messages we not give as a parameter then by default it will return string content as an output

### Interview Question : Without this conversational Buffer memory can you create your own memory?

#### --> YES, create a list, then we will loop the list and we will be going on append the human message and AI message, and that appended list i will be using inside prompt template.

- note : This all thing we are able to done by using ConversationBufferMemory class

In [ ]:
# Step 2 : Create a ChatpromptTemplate (cpt)

cpt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are a helpful assistance"),
                                        MessagesPlaceholder(variable_name = "storage"),
                                        HumanMessagePromptTemplate.from_template("{input}")])

# MessagesPlaceholder: It is also prompt template by using which we can insert a list of message inside your message (ChatPromptTemplate.from_messages).


In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
sto = StrOutputParser()

In [ ]:
chain =LLMChain(prompt = cpt ,
                llm = model,
                output_parser = sto ,
                memory = memory1)

In [ ]:
chain.run(input="Hi, How are you?")

/tmp/ipykernel_2305/368182495.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  chain.run(input="Hi, How are you?")


ValueError: variable storage should be a list of base messages, got  of type <class 'str'>

### Explain Error: variable storage should be a list of base messages, got  of type <class 'str'>
#### --> This Error occurred Because of when we calling chain this chain is calling an object (memory1):memory1 = ConversationBufferMemory(memory_key = "storage"). but the problem is when this memory is called is is not returning list of message, it is returning string , and that string can be appended inside the placeholder.and that placeholder oly accepts list of string

In [ ]:
chain.run(input="Hi, How are you?")

"Hi there! I'm doing great, thank you for asking!\n\nHow are you doing today?"

In [ ]:
chain.run(input="I am shrutika and you")

# Run work only when ConversationBufferMemory parameter return_messages is True in list format

"It's lovely to meet you, Shrutika!\n\nAs for me, I don't have a name like a person does. I'm an AI assistant, a large language model trained by Google. You can just call me AI, or assistant!"

In [ ]:
memory1.load_memory_variables({}) # Show the internal memory

{'storage': [HumanMessage(content='Hi, How are you?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hi there! I'm doing great, thank you for asking!\n\nHow are you doing today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='I am shrutika and you', additional_kwargs={}, response_metadata={}),
  AIMessage(content="It's nice to meet you, Shrutika!\n\nI don't have a name like a person does. I am a large language model, an AI, trained by Google. You can just call me AI or assistant!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='I am shrutika and you', additional_kwargs={}, response_metadata={}),
  AIMessage(content="It's lovely to meet you, Shrutika!\n\nAs for me, I don't have a name like a person does. I'm an AI assistant, a large language model trained by Google. You can just call me AI, or assistant!", additional_kwargs={}, response_metadata={},

In [ ]:
chain.run(input="What is machine learning")

'That\'s a fantastic question, Shrutika! Machine Learning (ML) is one of the most exciting and impactful fields in technology today.\n\nAt its core, **Machine Learning is a subset of Artificial Intelligence (AI) that enables computer systems to "learn" from data, identify patterns, and make decisions or predictions without being explicitly programmed for every specific task.**\n\nThink of it this way:\n\n*   **Traditional Programming:** You write a set of specific instructions for the computer to follow. If you want it to identify a cat, you\'d have to write code for every single feature of a cat (ears, whiskers, fur color, etc.) and all the variations. This is incredibly difficult and often impossible for complex tasks.\n\n*   **Machine Learning:** Instead of giving explicit instructions, you give the computer a *lot* of data (e.g., thousands of pictures labeled "cat" and "not cat"). The machine learning algorithm then figures out the patterns and rules on its own to differentiate bet

In [ ]:
memory1.load_memory_variables({})

# The below output acting as a context for the LLM

{'storage': [HumanMessage(content='Hi, How are you?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hi there! I'm doing great, thank you for asking!\n\nHow are you doing today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='I am shrutika and you', additional_kwargs={}, response_metadata={}),
  AIMessage(content="It's nice to meet you, Shrutika!\n\nI don't have a name like a person does. I am a large language model, an AI, trained by Google. You can just call me AI or assistant!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='I am shrutika and you', additional_kwargs={}, response_metadata={}),
  AIMessage(content="It's lovely to meet you, Shrutika!\n\nAs for me, I don't have a name like a person does. I'm an AI assistant, a large language model trained by Google. You can just call me AI, or assistant!", additional_kwargs={}, response_metadata={},

In [ ]:
chain.run(input="What is my name")

'Your name is **Shrutika**! You told me earlier.'

### So the above conversational Buffer Memory in a old fashion, Now the same thing i need to replicate and created with the help of runnables

#### Newer version of creating Conversationalbuffermemory

In [4]:
from langchain_core.runnables.history import RunnableWithMessageHistory # Wrapper
from langchain_core.chat_history import InMemoryChatMessageHistory #RedisChatMessageHistory #All the different type of session_store

# InMemoryChatMessageHistory: It is called local memory, when the sessional memory is over all the chat will be losed
# RedisChatMessageHistory: It is a CLOUD PLATFORM by using which we can create a cloud database
# Redis is a platform : Data stored in RAM → microsecond-level latency
# Much faster than disk-based DBs like MySQL
# Unlike traditional databases, Redis stores data in RAM, making it extremely fast

In [5]:
# Create a Session_store

storage ={}

def session_factory(session_id):
  if session_id not in storage:
   storage[session_id] = InMemoryChatMessageHistory()
  return storage[session_id]


In [6]:
#Create a ChatpromptTemplate (cpt)

cpt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are a helpful assistance"),
                                        MessagesPlaceholder(variable_name = "storage"),
                                        HumanMessagePromptTemplate.from_template("{input}")])

# MessagesPlaceholder: It is also prompt template by using which we can insert a list of message inside your message (ChatPromptTemplate.from_messages).


In [7]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [8]:
sto = StrOutputParser()

In [10]:
chain1 = cpt|model|sto

In [12]:
chain_with_memory = RunnableWithMessageHistory(runnable=chain1,get_session_history=session_factory, history_messages_key="storage", input_messages_key="input")

In [13]:
chain_with_memory

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  storage: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function session_factory at 0x79cb3a327f60>, input_messages_key='input', history_messages_key='storage', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [14]:
con ={"configurable":{"session_id":"person1"}}

In [15]:
con

{'configurable': {'session_id': 'person1'}}

In [16]:
chain_with_memory.invoke({"input":"hi"},config = con)

'Hi there! How can I help you today?'

In [17]:
chain_with_memory.invoke({"input":"My name is shrutika"},config = con)

'Nice to meet you, Shrutika! How can I help you today?'

In [18]:
chain_with_memory.invoke({"input":"What is my name"},config = con)

'Your name is Shrutika!'

In [19]:
chain_with_memory.invoke({"input":"What is my name"},config = {"configurable":{"session_id":"person2"}}) #It is not able to answer

"I don't know your name. As an AI, I don't have access to personal information about you.\n\nIf you'd like to tell me, I'll be happy to remember it for our conversation!"

In [21]:
storage.keys()

dict_keys(['person1', 'person2'])

In [22]:
storage

{'person1': InMemoryChatMessageHistory(messages=[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='My name is shrutika', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, Shrutika! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Shrutika!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]),
 'person2': InMemoryChatMessageHistory(messages=[HumanMessage(content='What is my name', additional_kwargs={}, response_metadata={}), AIMessage(content="I don't know your name. As an AI, I don't have access to personal information about you.\n\nIf you'd like to tell me, I'll be

In [ ]:
# We can create complate Conversational Buffer Memory with runnable and with one additional advantage: Multiple user can use but the same disadvantage is chat will increasing message is also increased.